# OrigenLM — fine-tuning QLoRA (Kaggle T4)

Adapta Qwen2.5-Coder-7B-Instruct a los diálogos de OrigenLM.

- **Settings → Accelerator: GPU T4 x2**, Internet **ON**.
- El repo se clona desde GitHub: antes tiene que estar subido a `main`.
- Al terminar, descargá `checkpoints/qlora_origen` desde *Output*.

In [ ]:
REPO = "https://github.com/J3lk5d0x3l/OrigenLM.git"
COMMIT = "main"

!git clone {REPO} /kaggle/working/OrigenLM
%cd /kaggle/working/OrigenLM
!git checkout {COMMIT}
!git log --oneline -1
!pip -q install -r requirements.txt

In [ ]:
!python scripts/preparar_sft.py
!python scripts/agregar_dialogos.py --seco
!python train_qlora.py --dry-run
!head -1 dataset/sft/train.jsonl

In [ ]:
# Una época sobre los ejemplos actuales: minutos en T4. Subí --epochs para repetir.
!python train_qlora.py \
  --epochs 1 \
  --batch 2 --grad-acum 8 \
  --max-len 512 \
  --eval-cada 50 \
  --salida checkpoints/qlora_origen

In [ ]:
import json, torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = "Qwen/Qwen2.5-Coder-7B-Instruct"
tok = AutoTokenizer.from_pretrained(base)
modelo = AutoModelForCausalLM.from_pretrained(base, dtype=torch.float16, device_map="auto")
modelo = PeftModel.from_pretrained(modelo, "checkpoints/qlora_origen/adapter")
modelo.eval()

mensajes = [{"role": "system", "content": "Eres Origen, un asistente claro y útil. Respondés en español."},
            {"role": "user", "content": "Hola, ¿quién eres?"}]
ids = tok.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt").to(modelo.device)
salida = modelo.generate(ids, max_new_tokens=120, temperature=0.3, do_sample=True,
                         pad_token_id=tok.eos_token_id)
print(tok.decode(salida[0][ids.shape[1]:], skip_special_tokens=True))
print(json.load(open("checkpoints/qlora_origen/meta.json")))

## Después

El adaptador pesa ~40 MB. Para usarlo en el codespace (CPU) conviene fusionarlo
y exportarlo a GGUF:

```bash
python train_qlora.py --fusionar        # deja checkpoints/qlora_origen/modelo_fusionado
python llama.cpp/convert_hf_to_gguf.py checkpoints/qlora_origen/modelo_fusionado --outfile models/origen-fusionado.gguf
```

Con pocos ejemplos el resultado aprende estilo, no conocimiento. Para mejorar de
verdad hay que sumar datos: ver `dataset/entradas/LEEME.md`.